### Cell 09.01 — recover the frozen days-to-flowering result

In [ ]:
# Cell 09.01
# Start Notebook 09:
# days_fl_07 suggestive QTL on pLG07 / candidate Gm08.

from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


QTL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)


all33 = pd.read_excel(
    QTL_FILE,
    sheet_name="all_33_empirical"
)


days_row = (
    all33
    .loc[
        all33["trait"] == "days_fl_07"
    ]
    .copy()
)


print("days_fl_07 rows:", len(days_row))

display(days_row)


DAYSFL_QTL = {
    "trait": "days_fl_07",
    "peak_marker": "TMA2",
    "structural_group": "pLG07",
    "candidate_chr": "Gm08",

    "lod": float(
        days_row.iloc[0]["lod"]
    ),

    "r2": float(
        days_row.iloc[0]["r2"]
    ),

    "n": int(
        days_row.iloc[0]["n"]
    ),

    "empirical_p": float(
        days_row.iloc[0]["peak_empirical_p"]
    ),

    "genomewide_status":
        days_row.iloc[0]["genomewide_status"],

    "physical_assignment":
        "insufficient_single_anchor",

    "peak_direct_physical_anchor":
        False
}


print()
print("FROZEN days_fl_07 QTL")
print("=" * 90)

for key, value in DAYSFL_QTL.items():
    print(f"{key}: {value}")

### Cell 09.02 — recover the full local QTL profile on pLG07
* We already know the neighboring marker effects were all in the same direction and increased toward TMA2. Let's now recover the exact table from the frozen QTL workbook.

In [ ]:
# Cell 09.02
# Recover the frozen regional profile for days_fl_07.

days_region = pd.read_excel(
    QTL_FILE,
    sheet_name="region_days_fl_07"
)


print(
    "Region table shape:",
    days_region.shape
)

print(
    "Columns:"
)

print(
    days_region.columns.tolist()
)


display(days_region)

### Cell 09.03 — inspect all pLG07 framework markers and physical anchor evidence

In [ ]:
# Cell 09.03
# Recover pLG07 framework markers and all independent physical evidence.

MAP_FILE = (
    PROJECT_ROOT
    / "results"
    / "linkage_map"
    / "flyer_hartwig_structural_physical_map_final.xlsx"
)


map_xls = pd.ExcelFile(
    MAP_FILE
)


print("Map sheets:")
print(map_xls.sheet_names)


ordered = pd.read_excel(
    MAP_FILE,
    sheet_name="ordered_framework"
)


anchor_evidence = pd.read_excel(
    MAP_FILE,
    sheet_name="anchor_evidence"
)


physical_votes = pd.read_excel(
    MAP_FILE,
    sheet_name="physical_votes"
)


plg07_ordered = (
    ordered
    .loc[
        ordered["structural_group"]
        == "pLG07"
    ]
    .copy()
)


print()
print("pLG07 ordered markers:")
print("=" * 90)

display(plg07_ordered)


print()
print("ANCHOR EVIDENCE COLUMNS")
print("=" * 90)

print(anchor_evidence.columns.tolist())


# Search all anchor-evidence rows involving any pLG07 marker.
plg07_markers = set(
    plg07_ordered["marker"]
    .astype(str)
)


marker_col_candidates = [
    c
    for c in anchor_evidence.columns
    if "marker" in str(c).lower()
]


print()
print(
    "Possible marker columns:",
    marker_col_candidates
)

### Cell 09.04 — identify the single independent anchor behind the Gm08 assignment
* This cell is deliberately robust to slightly different column names in the final workbook.

In [ ]:
# Cell 09.04
# Find all physical-evidence records corresponding to pLG07 markers.

possible_marker_columns = [
    c
    for c in anchor_evidence.columns
    if (
        "marker" in str(c).lower()
        or str(c).lower() == "name"
    )
]


plg07_anchor_rows = pd.DataFrame()


for col in possible_marker_columns:

    tmp = anchor_evidence.loc[
        anchor_evidence[col]
        .astype(str)
        .isin(plg07_markers)
    ].copy()

    if len(tmp) > 0:

        tmp[
            "_matched_marker_column"
        ] = col

        plg07_anchor_rows = pd.concat(
            [
                plg07_anchor_rows,
                tmp
            ],
            ignore_index=True
        )


plg07_anchor_rows = (
    plg07_anchor_rows
    .drop_duplicates()
    .reset_index(drop=True)
)


print(
    "Physical anchor rows associated with pLG07:",
    len(plg07_anchor_rows)
)


display(plg07_anchor_rows)


print()
print("Physical vote records involving pLG07:")
print("=" * 90)


# Show relevant vote rows using any likely group column.
group_cols = [
    c
    for c in physical_votes.columns
    if (
        "group" in str(c).lower()
        or "plg" in str(c).lower()
    )
]


vote_subset = pd.DataFrame()


for col in group_cols:

    tmp = physical_votes.loc[
        physical_votes[col]
        .astype(str)
        .eq("pLG07")
    ].copy()

    if len(tmp) > 0:

        vote_subset = pd.concat(
            [
                vote_subset,
                tmp
            ],
            ignore_index=True
        )


vote_subset = (
    vote_subset
    .drop_duplicates()
    .reset_index(drop=True)
)


display(vote_subset)

### Cell 09.05 — quantify TMA2-to-anchor linkage distance

In [ ]:
# Cell 09.05
# Quantify how closely the QTL peak is linked to the sole
# independent physical anchor on pLG07.

TMA2_ROW = (
    plg07_ordered
    .loc[
        plg07_ordered["marker"] == "TMA2"
    ]
    .iloc[0]
)

SAT162_ROW = (
    plg07_ordered
    .loc[
        plg07_ordered["marker"] == "Sat_162"
    ]
    .iloc[0]
)


TMA2_CM = float(
    TMA2_ROW[
        "kosambi_cm_structural_provisional"
    ]
)

SAT162_CM = float(
    SAT162_ROW[
        "kosambi_cm_structural_provisional"
    ]
)

TMA2_TO_SAT162_CM = abs(
    SAT162_CM - TMA2_CM
)


SAT162_BP = int(
    plg07_anchor_rows.iloc[0][
        "physical_bp"
    ]
)


print("days_fl_07 PHYSICAL-ANCHOR CONTEXT")
print("=" * 90)

print(
    f"TMA2 position:        {TMA2_CM:.6f} provisional cM"
)

print(
    f"Sat_162 position:     {SAT162_CM:.6f} provisional cM"
)

print(
    f"Genetic separation:   {TMA2_TO_SAT162_CM:.6f} provisional cM"
)

print(
    f"Sat_162 physical bp:  {SAT162_BP:,}"
)

print(
    "Sat_162 chromosome:   Gm08"
)

print(
    "Anchor evidence:      exact"
)

print()
print(
    "IMPORTANT: the genetic distance cannot be converted "
    "into a physical Mb interval from a single anchor."
)

### Cell 09.06 — summarize the regional allele-effect pattern

In [ ]:
# Cell 09.06
# Summarize the regional phenotype pattern.
#
# Genotype coding:
#   2 = Flyer
#   0 = Hartwig
#
# Therefore negative effect_2_minus_0 means
# the Flyer allele reduces days to flowering.

days_region_summary = (
    days_region[
        [
            "marker",
            "structural_order",
            "kosambi_cm_structural_provisional",
            "mean_genotype_0",
            "mean_genotype_2",
            "effect_2_minus_0",
            "lod"
        ]
    ]
    .copy()
)


days_region_summary[
    "favorable_direction"
] = np.where(
    days_region_summary[
        "effect_2_minus_0"
    ] < 0,
    "Flyer allele -> earlier flowering",
    "Hartwig allele -> earlier flowering"
)


print("REGIONAL EFFECT PATTERN")
print("=" * 90)

print(
    "Markers with negative Flyer-Hartwig effect:",
    int(
        (
            days_region_summary[
                "effect_2_minus_0"
            ] < 0
        ).sum()
    ),
    "/",
    len(days_region_summary)
)


peak_effect = (
    days_region_summary
    .loc[
        days_region_summary[
            "marker"
        ] == "TMA2"
    ]
    .iloc[0]
)


print()
print("TMA2 peak:")
print(
    f"  Hartwig mean (0): "
    f"{peak_effect['mean_genotype_0']:.3f} days"
)

print(
    f"  Flyer mean (2):   "
    f"{peak_effect['mean_genotype_2']:.3f} days"
)

print(
    f"  Flyer - Hartwig:  "
    f"{peak_effect['effect_2_minus_0']:.3f} days"
)


display(days_region_summary)

### Cell 09.07 — create a conservative locus interpretation

In [ ]:
# Cell 09.07
# Freeze the interpretation without inventing a physical QTL interval.

days_fl_interpretation = pd.DataFrame(
    [
        {
            "trait":
                "days_fl_07",

            "peak_marker":
                "TMA2",

            "structural_group":
                "pLG07",

            "candidate_chr":
                "Gm08",

            "lod":
                float(
                    days_row.iloc[0]["lod"]
                ),

            "r2":
                float(
                    days_row.iloc[0]["r2"]
                ),

            "n":
                int(
                    days_row.iloc[0]["n"]
                ),

            "empirical_p":
                float(
                    days_row.iloc[0][
                        "peak_empirical_p"
                    ]
                ),

            "genomewide_status":
                "suggestive_10pct",

            "physical_assignment":
                "insufficient_single_anchor",

            "sole_physical_anchor":
                "Sat_162",

            "sole_anchor_chr":
                "Gm08",

            "sole_anchor_bp_gnm6":
                SAT162_BP,

            "sole_anchor_evidence":
                "exact",

            "TMA2_to_anchor_cm":
                TMA2_TO_SAT162_CM,

            "peak_effect_2_minus_0_days":
                float(
                    peak_effect[
                        "effect_2_minus_0"
                    ]
                ),

            "effect_interpretation":
                (
                    "Flyer allele associated with "
                    "earlier flowering"
                ),

            "regional_direction_consistent":
                bool(
                    (
                        days_region[
                            "effect_2_minus_0"
                        ] < 0
                    ).all()
                ),

            "physical_candidate_gene_analysis":
                "not_performed",

            "reason_no_candidate_gene_window":
                (
                    "TMA2 has no direct physical anchor and "
                    "pLG07 contains only one independently "
                    "anchored marker. A single anchor cannot "
                    "define physical direction or boundaries "
                    "for the QTL."
                )
        }
    ]
)


display(
    days_fl_interpretation.T
)

### Cell 09.08 — export and freeze Notebook 09

In [ ]:
# Cell 09.08
# Export final conservative days_fl_07 summary.

DAYS_FINAL_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_days_fl07_gm08_context.xlsx"
)


with pd.ExcelWriter(
    DAYS_FINAL_FILE,
    engine="openpyxl"
) as writer:

    days_fl_interpretation.to_excel(
        writer,
        sheet_name="locus_summary",
        index=False
    )

    days_region_summary.to_excel(
        writer,
        sheet_name="regional_profile",
        index=False
    )

    plg07_ordered.to_excel(
        writer,
        sheet_name="pLG07_framework",
        index=False
    )

    plg07_anchor_rows.to_excel(
        writer,
        sheet_name="physical_anchor",
        index=False
    )

    vote_subset.to_excel(
        writer,
        sheet_name="physical_votes",
        index=False
    )


print("FINAL days_fl_07 EXPORT")
print("=" * 90)
print(DAYS_FINAL_FILE)

print()
print("Notebook 09 can now be frozen.")

print()
print(
    "No candidate-gene window was generated because "
    "the peak marker lacks a direct physical coordinate "
    "and pLG07 has only one independent physical anchor."
)